# 随机森林用来计算特征重要修改。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier

# 朴素贝叶斯

In [ ]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.datasets import load_iris
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
import glob
from pathlib import Path

## 贝叶斯公式

朴素贝叶斯算法是一个典型的统计学习方法。

主要理论基础是**贝叶斯公式**：

$\huge P(Y_k|X) = \frac{P(XY_k)}{P(X)} = \frac{P(Y_k)P(X|Y_k)}{\sum_j P(Y_j)P(X|Y_j)}$

$
其中:\\
P(Y_k|X)：在特征为X的条件下，结果Y为情况k的概率。\\
P(XY_k)：特征为X且结果Y为情况k的概率。\\
P(X)：特征X出现的概率。\\
P(Y_k)：结果Y为情况k的概率。\\
P(X|Y_k)：在结果Y为情况k的条件下，特征为X的概率。\\
j：结果Y的其中一种情况。\\
\sum_j：对后面的表达式，将每种结果j计算的概率累加。
$

这个公式看上去简单，但它却能总结历史，预知未来：  
- 公式的右边是总结历史。
- 公式的左边是预知未来。

也就是说我们想知道在某种特征出现情况下，结果Y为k的概率。那么只需要根据过去的数据统计结果Y为k的概率，结果Y为k情况下特征为X的概率，以及结果Y的每类结果的以上概率。

### 小球案例

有A桶和B桶，A桶里装了3个红球，6个黑球，B桶里装了1个红球，7个黑球。  
现在随机从两个桶里取1个球，结果是红球。  
问：这个红球来自A桶的概率是多少？

以R表示抽到的球为红色，A表示红球属于A桶，B表示球属于B桶，根据贝叶斯公式：

$P(A|R) = \frac{P(R|A) \cdot P(A)}{P(R)} $

$
P(R) = P(R|A) \cdot P(A) + P(R|B) \cdot P(B)\\
 = \left( \frac{1}{3} \cdot \frac{1}{2} \right) + \left( \frac{1}{8} \cdot \frac{1}{2} \right) \\
 = \frac{1}{6} + \frac{1}{16}\\
 = \frac{8}{48} + \frac{3}{48}\\
 = \frac{11}{48}
$

$
P(A) = \frac{1}{2} 
$

$
P(R|A) = \frac{3}{9} = \frac{1}{3}
$

$
\large 所以，P(A|R) = \frac{\frac{1}{3} \cdot \frac{1}{2}}{\frac{11}{48}} = \frac{8}{11}
$

### 图书馆案例

大学的时候，某男生经常去图书室晚自习，发现他喜欢的那个女生也常去那个自习室，心中窃喜，于是每天买点好吃点在那个自习室蹲点等她来，可是人家女生不一定每天都来，眼看天气渐渐炎热，图书馆又不开空调，如果那个女生没有去自修室，该男生也就不去，每次男生鼓足勇气说：“嘿，你明天还来不？”,“啊，不知道，看情况”。

然后该男生每天就把她去自习室与否以及一些其他情况做一下记录，用Y表示该女生是否去自习室，即Y={去，不去}，X是跟去自修室有关联的一系列条件，比如当天上了哪门主课，蹲点统计了一段时间后，该男生打算今天不再蹲点，而是先预测一下她会不会去，现在已经知道了今天上了常微分方法这么主课，于是计算P(Y=去|常微分方程)与P(Y=不去|常微分方程)，看哪个概率大，如果P(Y=去|常微分方程) >P(Y=不去|常微分方程)，那这个男生不管多热都屁颠屁颠去自习室了，否则不就去自习室受罪了。P(Y=去|常微分方程)的计算可以转为计算以前她去的情况下，那天主课是常微分的概率P(常微分方程|Y=去)，注意公式右边的分母对每个类别（去/不去）都是一样的，所以计算的时候忽略掉分母，这样虽然得到的概率值已经不再是0~1之间，但是通过比较大小还是能选择类别。

后来他发现还有一些其他条件可以挖，比如当天星期几、当天的天气，以及上一次与她在自修室的气氛，统计了一段时间后，该男子一计算，发现不好算了，因为总结历史的公式：

$
\large P(X=x|Y=c_k) = P(X^{(1)}=x^{(1)}, ... , X^{(n)}=x^{(n)} | Y=c_k)
$

这里n=4，x(1)表示主课，x(2)表示天气，x(3)表示星期几，x(4)表示气氛，Y仍然是{去，不去}，现在主课有8门，天气有晴、雨、阴三种、气氛有A+,A,B+,B，C五种，那么总共需要估计的参数有8×3×7×5×2=1680个，每天只能收集到一条数据，那么等凑齐1680条数据，大学都毕业了，男生大呼不妙，于是做了一个独立性假设，假设这些影响她去自习室的原因是独立互不相关的，于是：

$
\large P(X=x|Y=c_k) = \prod\limits_{j=1}^{n}P(X{(j)}=x^{(j)} | Y=c_k)
$

有了这个独立假设后，需要估计的参数就变为，(8+3+7+5)×2 = 46个了，而且每天收集的一条数据，可以提供4个参数，这样该男生就预测越来越准了。

**朴素的概念：独立性假设，假设各个特征之间是独立不相关的。**

## 朴素贝叶斯分类器

讲了上面的小故事，我们来朴素贝叶斯分类器的表示形式：

$
\huge y = f(x) = arg\underset{c_k}{max} \frac{P(Y=c_k) \prod\limits_{j}P(X^{(j)} = x^{(j)} | Y=c_k)}{\sum\limits_{k} P(Y=c_k) \prod\limits_{j}P(X^{(j)} = x^{(j)} | Y=c_k}
$

当特征为为x时，计算所有类别的条件概率，选取条件概率最大的类别作为待分类的类别。由于上公式的分母对每个类别都是一样的，因此计算时可以不考虑分母，即

$
\huge y = f(x) = arg\underset{c_k}{max} P(Y=c_k) \prod\limits_{j}P(X^{(j)} = x^{(j)} | Y=c_k)
$

朴素贝叶斯的朴素体现在其对各个条件的独立性假设上，加上独立假设后，大大减少了参数假设空间。

## 文本分类

文本分类的应用很多，比如垃圾邮件和垃圾短信的过滤就是一个2分类问题，新闻分类、文本情感分析等都可以看成是文本分类问题，分类问题由两步组成：训练和预测，要建立一个分类模型，至少需要有一个训练数据集。贝叶斯模型可以很自然地应用到文本分类上：现在有一篇文档d（Document），判断它属于哪个类别ck，只需要计算文档d属于哪一个类别的概率最大：

$
\large c = arg \underset{c_k}{max} P(c_k|d)
$

在分类问题中，我们并不是把所有的特征都用上，对一篇文档d，我们只用其中的部分特征词项t1,t2,...,tnd（nd表示d中的总词条数目），因为很多词项对分类是没有价值的，比如一些停用词“的,是,在”在每个类别中都会出现，这个词项还会模糊分类的决策面，关于特征词的选取，我的这篇文章有介绍。用特征词项表示文档后，计算文档d的类别转化为：

$
\large c = arg \underset{c_k}{max} P(c_k) \prod\limits_{1 \le j \le n} P(t_j|c_k)
$

注意P(Ck|d)只是正比于后面那部分公式，完整的计算还有一个分母，但我们前面讨论了，对每个类别而已分母都是一样的，于是在我们只需要计算分子就能够进行分类了。实际的计算过程中，多个概率值P(tj|ck)的连乘很容易下溢出为0，因此转化为对数计算，连乘就变成了累加：

$
\large c = arg \underset{c_k}{max}[log P(c_k) + \sum\limits_{1 \le j \le n} log P(t_j|c_k)]
$

我们只需要从训练数据集中，计算每一个类别的出现概率P(ck)和每一个类别中各个特征词项的概率P(tj|ck)，而这些概率值的计算都采用最大似然估计，说到底*就是统计每个词在各个类别中出现的次数和各个类别的文档的数目*：

$
\large P(c_k) = \frac{N_{c_k}}{N} \\
\large P(t_j|c_k) = \frac{T_{jk}}{\sum\limits_{1 \le i \le |V|} T_{ik}}
$

$
其中:\\
N_{c_k}：类别k文档的个数。\\
N：总文档个数。\\
T_{jk}：类别k文档里特征词j出现的次数。\\
T_{IK}：类别k文档里特征词i出现的次数，前面的累加符号表示i取所有特征词情况下的累加和。
$

## 三种朴素贝叶斯分类器

**根据特征的分布规律不同，有三种朴素贝叶斯分类器**

### 高斯分布朴素贝叶斯

特征值分布符合高斯分布。自然界大部分特征都是这种情况，离某个中心值越远出现概率越小。

In [ ]:
iris = load_iris()
data = iris.data
target = iris.target

In [ ]:
GaussianNB().fit(data, target).score(data, target)

### 多项式分布朴素贝叶斯

特征符合多项式分布。通常用于特征为离散，固定值，非负整数的情况，如词频统计。

$
多项式分布：特征向量 (X = (X_1, X_2, \ldots, X_k)) 表示 (k) 种不同结果的次数，其中 (X_i) 表示结果 (i) 出现的次数。这些次数是离散的非负整数，并且满足：
[ X_1 + X_2 + \cdots + X_k = n ]
$

$
每个特征值 (X_i) 的概率由下式给出：\\
[ P(X_1 = x_1, X_2 = x_2, \ldots, X_k = x_k) = \frac{n!}{x_1! x_2! \cdots x_k!} p_1^{x_1} p_2^{x_2} \cdots p_k^{x_k} ]\\
其中：\\
(n) 是试验次数，\\
(x_i) 是结果 (i) 出现的次数，\\
(p_i) 是结果 (i) 发生的概率，满足 ( \sum_{i=1}^{k} p_i = 1 )。\\
$

In [ ]:
MultinomialNB().fit(data, target).score(data, target)

### 伯努利分布

假设特征向量 ( X ) 的每个元素是相互独立的，并且每个元素的取值符合伯努利分布。即每个特征要么出现（1），要么不出现（0），适用于特征是二值的情况。

也就是说，每个特征要么是1，要么是0，只有两种情况。（这里不要求两种情况的概率相等。）

常见应用：垃圾邮件检测（是否包含某些敏感词），情感分析（是否包含特定情感词）

In [ ]:
BernoulliNB().fit(data, target).score(data, target)

### 三种分类器总结

从上面三种分类的特征分布要求来看，鸢尾花明显是属于高斯分布朴素贝叶斯适用的场景，但多项式也能处理（数据量越大，特征值越多会不会表现不好？），伯努利明显是不合适。

画出三种分类器的分界线

In [ ]:
# 选取前两个特征作为分界依据。
# 根据两个特征的最小值到最大值，在两个特征范围内取1000个值。
x = np.linspace(data[:, 0].min(), data[:, 0].max(), 1000)
y = np.linspace(data[:, 1].min(), data[:, 1].max(), 1000)

# 使用meshgrid根据x，y，生成交叉点的横坐标数组和纵坐标数组。
X, Y = np.meshgrid(x, y)

# 横坐标和纵坐标扁平化为一维数组后，使用c_将数组转为一列的二维数组并按列方向拼接，得到交叉点的坐标。
XY = np.c_[X.ravel(), Y.ravel()]
display(x.min(), x.max(), y.min(), y.max(), X.shape, X.shape, XY)

In [ ]:
# 使用模型预测这些坐标点。
y_ = GaussianNB().fit(data[:, :2], target).predict(XY)

In [ ]:
# 使用pcolormesh画出分类边界线.参数分别为X坐标值，Y坐标值，预测结果reshape为网格形状。shading='auto'避免警告
plt.pcolormesh(X, Y, y_.reshape(1000, 1000), shading='auto')
# 画出特征点的分布，x为特征1数据，y为特征2数据，颜色为target，颜色空间为rainbow，防止被算法的背景色覆盖。
plt.scatter(data[:, 0], data[:, 1], c=target, cmap='rainbow')

In [ ]:
# 定义方法：接受模型对象，绘制模型对象的分类边界。
def draw_border(model):
    # 使用模型预测这些坐标点。
    y_ = model.fit(data[:, :2], target).predict(XY)
    # 使用pcolormesh画出分类边界线.参数分别为X坐标值，Y坐标值，预测结果reshape为网格形状。shading='auto'避免警告
    plt.pcolormesh(X, Y, y_.reshape(1000, 1000), shading='auto')
    # 画出特征点的分布，x为特征1数据，y为特征2数据，颜色为target，颜色空间为rainbow，防止被算法的背景色覆盖。
    plt.scatter(data[:, 0], data[:, 1], c=target, cmap='rainbow')

In [ ]:
draw_border(GaussianNB())

In [ ]:
draw_border(MultinomialNB())

In [ ]:
draw_border(BernoulliNB())

**伯努利分布朴素贝叶斯分类器，要求所有特征都是0或1。  
鸢尾花数据明显不符合要求，算法将所有不为0的特征值都视为1，所有样本特征都为[1, 1, 1, 1]，因此预测结果只会是其中一种。**

**从上面的算法分类界限图可以看出，对于常见的分类问题，一般使用高斯分布比较合适。**

## 文本分类实战

### 垃圾短信分类

In [ ]:
sms = pd.read_csv('../day33_朴素贝叶斯(二)/代码/data/SMSSpamCollection', sep='\t', header=None)
sms

第1列为短信分类，spam表示营销推广，骚扰，诈骗等内容，ham表示正常内容。

第2列为短信内容

In [ ]:
sms.iloc[0, 1]

这句话的中文翻译是：
去到裕廊点（Jurong Point），疯狂的.. 只在牛车水（Bugis）和大世界（Great World）有自助餐... 那里的电影院有更多选择...
注：这是典型的新加坡式英语（Singlish）表达，带有口语化和省略语法的特点。

In [ ]:
sms.iloc[2, 1]

这条短信的中文翻译是：
免费参加每周2次的比赛，赢取2005年5月21日足总杯决赛门票。发送FA到87121获取参赛问题（标准短信费率）。条款和条件适用。08452810075，仅限18岁以上人员参加。
这是一条典型的英国促销短信广告，内容是关于足总杯（FA Cup）决赛门票的抽奖活动。

对短信做垃圾分类，需要先把短信内容转换为数值，然后使用朴素贝叶斯模型进行训练和预测。

文本转数字的算法有很多，这里介绍最原始的两种算法，one-hot编码和tfidf

#### one-hot编码

one-hot编码的原理非常简单，就是进行词频统计，将全部样本中的词汇去重，排序后建立词表，每个词映射为数字。  
使用词表作为特征列，如果词表当前位置的词在样本里出现，那么当前位置记录为出现的次数，没有出现则记为0。  
最终得到的特征数据是一个稀疏矩阵（大部分值都为0的矩阵）。

In [ ]:
# CountVectorizer就是使用one-hot编码做转换
cVector = CountVectorizer()
data_vec = cVector.fit_transform(sms.iloc[:, 1].ravel()).toarray()

In [ ]:
data_vec

In [ ]:
# 每个样本去重后的词数。
data_vec.sum(axis=1)

In [ ]:
# 可以查看构建的词频表。
word_dict = dict(sorted(cVector.vocabulary_.items()))

In [ ]:
# 取出第一个样本词频不为0的下标和值。
s = pd.Series(data=data_vec[0][data_vec[0] != 0], index=np.nonzero(data_vec[0])[0])
s

In [ ]:
# 从词表里查看这些下标对应的词汇
reverse_dict = {v:k for k, v in word_dict.items()}
s2 = pd.Series(reverse_dict)
s2[s.index]

In [ ]:
sms.iloc[0, 1]

可以看到通过模型词频表确实可以得到原始文本和转换后向量的关系。

对比原始文本和处理结果，可以看到大概的处理是文本统一转小写，用空格分隔单词，忽略逗号，点号，单个字母。

#### tf-idf



tfidf = term frequency 词频,  idf: inverse document frequency 逆文档频率

tf：单词在本文档出现的频率。idf：语料库文档总数/出现当前词的文档数。

tf和idf相乘，得到最终的词频结果。idf的作用是减小常见的to，and，or等词汇影响。

In [ ]:
tVerctor = TfidfVectorizer()
data_vec2 = tVerctor.fit_transform(sms.iloc[:, 1].ravel()).toarray()

#### 分类

In [ ]:
# 先使用one-hot编码处理数据
data_vec_count = CountVectorizer().fit_transform(sms.iloc[:, 1].ravel()).toarray()
data_vec_count.shape

In [ ]:
# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(data_vec_count, sms.iloc[:, 0].values)

In [ ]:
# 高斯分布贝叶斯
GaussianNB().fit(X_train, y_train).score(X_test, y_test)

In [ ]:
# 多项式分布贝叶斯
MultinomialNB().fit(X_train, y_train).score(X_test, y_test)

In [ ]:
# 伯努利分布贝叶斯
BernoulliNB().fit(X_train, y_train).score(X_test, y_test)

可以看出高斯分布表现比较差。（可能文本量很大时高斯分布会比较好？大样本场景下可能会更符合高斯分布）

In [ ]:
# 使用tfidf
data_vec_tfidf = TfidfVectorizer().fit_transform(sms.iloc[:, 1].ravel()).toarray()
data_vec_tfidf.shape

In [ ]:
# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(data_vec_tfidf, sms.iloc[:, 0].values)

In [ ]:
# 高斯分布贝叶斯
GaussianNB().fit(X_train, y_train).score(X_test, y_test)

In [ ]:
# 多项式分布贝叶斯
MultinomialNB().fit(X_train, y_train).score(X_test, y_test)

In [ ]:
# 伯努利分布贝叶斯
BernoulliNB().fit(X_train, y_train).score(X_test, y_test)

同样高斯分布表现比较差，伯努利分布在小样本场景下好像表现比多项式好一点。

## 作业

### 垃圾邮件分类

In [ ]:
# 读取数据
def read_content(typeDir):
    # 目录下文件列表
    files = list(Path(f"../day33_朴素贝叶斯(二)/代码/data/email/{typeDir}/").glob("*.txt"))
    # 根据文件名排序
    sorted_files = sorted(files, key=lambda x: int(x.name.replace('.txt', '')))
    # 读取文件内容（errors='ignore'是因为内容里有非utf-8字符，直接忽略处理）
    data = [(typeDir + '_' + f.name[:-4], Path(f).read_text(encoding='utf-8', errors='ignore')) for f in sorted_files]
    # 生成DataFrame
    df = pd.DataFrame(data, columns=['no', 'content'])
    df['target'] = typeDir
    return df

In [ ]:
df = pd.concat((read_content('spam'), read_content('ham')))
df

In [ ]:
data = df.iloc[:, 1].values
target = df.iloc[:, -1].values

In [ ]:
# 使用tfidf将文本转为向量
data_vec = TfidfVectorizer().fit_transform(data.ravel()).toarray()

In [ ]:
# 划分测试集和训练集
X_train, X_test, y_train, y_test = train_test_split(data_vec, target, test_size=0.2)
display(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
# 高斯分布贝叶斯
GaussianNB().fit(X_train, y_train).score(X_test, y_test)

In [ ]:
# 多项式分布贝叶斯
MultinomialNB().fit(X_train, y_train).score(X_test, y_test)

In [ ]:
# 伯努利分布贝叶斯
BernoulliNB().fit(X_train, y_train).score(X_test, y_test)

样本量太小了，评分很容易失真。